# 💻 Laboratório Prático: Validação e Pipelines de Dados
**OxeTech Academy**

---

## 📌 Sobre esta Atividade
Bem-vindo ao laboratório prático da Aula 04! O objetivo aqui é aplicar os conceitos de particionamento honesto, validação cruzada e pipelines que vimos na teoria. **Esta atividade é um treinamento livre (não avaliativo)**, mas é o aquecimento essencial para a AV1-04 e para o Projeto Final!

### 🎯 O que vamos praticar:
* **Data Splitting com Stratify:** Divisão correta que preserva a proporção das classes.
* **Pipeline com ColumnTransformer:** Pré-processamento encapsulado que previne Data Leakage.
* **Validação Cruzada Estratificada:** Estimativa honesta do desempenho com `StratifiedKFold`.
* **Comparação de estratégias:** K-Fold simples vs. Stratified — vendo a diferença na prática.

### 🗄️ Dataset: Titanic
Trabalharemos com o famoso dataset do Titanic. Ele contém informações dos passageiros (classe, sexo, idade, tarifa, porto de embarque) e se sobreviveram ou não (`survived`). É um problema de classificação binária — perfeito para praticar validação cruzada!

### ⚙️ Como Funciona a Dinâmica:
1. **Diagnóstico (🔍):** Algumas células servem apenas para mostrar o "problema" nos dados originais. Rode-as para entender o contexto.
2. **Sua Vez (👨‍💻):** Onde você encontrar o comentário `# --- SEU CÓDIGO AQUI ---`, substitua os valores `None` pela sua implementação.
3. **Checkpoints (🛑):** Após o seu código, sempre haverá um validador automático. Rode a célula para receber um feedback imediato (✅ Sucesso ou ❌ Erro) e saber se pode avançar para o próximo passo.

*Respire fundo, confie no processo e bom código!* 🚀

---

## PASSO 1: Preparação e Leitura dos Dados
Vamos importar as bibliotecas e carregar o dataset Titanic.
Apenas rode a célula abaixo para iniciar e conhecer nossos dados!

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, cross_val_score,
    StratifiedKFold, KFold
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Carregando o Titanic via seaborn
df_raw = sns.load_dataset('titanic').dropna(subset=['survived'])

# Selecionar colunas úteis para este exercício
colunas = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df_raw[colunas].copy()

print("✅ Dataset carregado com sucesso!")
print(f"   Shape: {df.shape}")
print(f"\n📊 Informações gerais:")
print(df.info())
print(f"\n🔍 Primeiras linhas:")
df.head()

In [ ]:
# Separando features e target
X = df.drop(columns=['survived'])
y = df['survived']

# Identificar tipos de colunas
cols_num = ['age', 'fare', 'sibsp', 'parch']
cols_cat = ['pclass', 'sex', 'embarked']

print("📋 Features numéricas:", cols_num)
print("📋 Features categóricas:", cols_cat)
print(f"\n🎯 Distribuição do target:")
print(y.value_counts())
print(f"\nProporção de sobreviventes: {y.mean():.1%}")
print("\n💡 Note: dataset desbalanceado (~38% sobreviveram) → usaremos stratify!")

## PASSO 2: Divisão Treino e Teste com Stratify

### 🔍 DIAGNÓSTICO: O Problema
Temos ~38% de sobreviventes. Se o split for aleatório puro, alguns folds podem ter proporções muito diferentes — distorcendo a avaliação.

⚠️ **INSTRUÇÃO:** Faça o split usando `train_test_split`. Use `test_size=0.2`, `random_state=42` e **não se esqueça do `stratify=y`** para preservar a proporção de classes. Substitua os `None` abaixo.

In [ ]:
# --- SEU CÓDIGO AQUI ---

# Use train_test_split com test_size=0.2, random_state=42 e stratify=y
X_train, X_test, y_train, y_test = None

# --- FIM DO CÓDIGO ---

# --- CHECKPOINT 1 ---
try:
    assert X_train is not None, "Faça o split antes de continuar!"
    assert len(X_train) > 0 and len(X_test) > 0, "Os conjuntos não podem estar vazios!"
    prop_train = y_train.mean()
    prop_test  = y_test.mean()
    prop_total = y.mean()
    assert abs(prop_train - prop_total) < 0.02,         f"Proporção do treino ({prop_train:.2%}) muito diferente do total ({prop_total:.2%}). Use stratify=y!"
    assert abs(prop_test - prop_total) < 0.02,         f"Proporção do teste ({prop_test:.2%}) muito diferente do total ({prop_total:.2%}). Use stratify=y!"
    print("✅ CHECKPOINT 1 PASSOU! Split estratificado realizado com sucesso.")
    print(f"   Treino: {len(X_train)} amostras ({prop_train:.1%} sobreviventes)")
    print(f"   Teste:  {len(X_test)} amostras ({prop_test:.1%} sobreviventes)")
    print(f"   Total:  {len(X)} amostras ({prop_total:.1%} sobreviventes)")
    print("\n💡 As proporções foram preservadas em treino e teste!")
except Exception as e:
    print(f"❌ ERRO NO CHECKPOINT 1: {e}")

## PASSO 3: Construção do Pipeline com ColumnTransformer

### 🔍 DIAGNÓSTICO: O Problema
Nossas features numéricas têm valores ausentes (especialmente `age`) e escalas diferentes. Nossas features categóricas precisam de encoding. **Tudo isso precisa ser encapsulado no Pipeline** — se fizermos fora, criamos leakage!

Rode a célula abaixo para visualizar o problema:

In [ ]:
print("=== VALORES AUSENTES EM X_train ===")
print(X_train.isnull().sum())
print()
print("=== ESCALAS DAS FEATURES NUMÉRICAS ===")
X_train[cols_num].describe().round(2).loc[['mean', 'std', 'max']]

⚠️ **INSTRUÇÃO:** Construa o Pipeline completo:
1. Um sub-pipeline para numéricas: `SimpleImputer(strategy='median')` → `StandardScaler()`
2. Um sub-pipeline para categóricas: `SimpleImputer(strategy='most_frequent')` → `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`
3. Um `ColumnTransformer` combinando os dois
4. O `pipeline` final: `preprocessor` → `LogisticRegression(max_iter=1000, random_state=42)`

In [ ]:
# --- SEU CÓDIGO AQUI ---

# 1. Sub-pipeline para colunas NUMÉRICAS
pipe_num = None

# 2. Sub-pipeline para colunas CATEGÓRICAS
pipe_cat = None

# 3. ColumnTransformer — combina os dois sub-pipelines
preprocessor = None

# 4. Pipeline final: preprocessor + classificador
pipeline = None

# --- FIM DO CÓDIGO ---

# --- CHECKPOINT 2 ---
try:
    assert pipe_num is not None, "Instancie pipe_num antes de continuar!"
    assert pipe_cat is not None, "Instancie pipe_cat antes de continuar!"
    assert preprocessor is not None, "Instancie o ColumnTransformer antes de continuar!"
    assert pipeline is not None, "Instancie o Pipeline final antes de continuar!"

    from sklearn.pipeline import Pipeline as _P
    from sklearn.compose import ColumnTransformer as _CT
    assert isinstance(pipeline, _P), "pipeline deve ser uma instância de Pipeline!"
    assert isinstance(preprocessor, _CT), "preprocessor deve ser um ColumnTransformer!"

    steps = [s[0] for s in pipeline.steps]
    assert 'preprocessor' in steps, "O Pipeline deve ter um passo chamado 'preprocessor'!"
    assert 'clf' in steps or any('logistic' in str(s).lower() or 'clf' in s for s in steps),         "O Pipeline deve ter um classificador!"

    print("✅ CHECKPOINT 2 PASSOU! Pipeline construído corretamente.")
    print(f"\nEstrutura do Pipeline:")
    print(pipeline)
    print("\n💡 Agora todos os transformadores estão encapsulados.")
    print("   O fit() só vai acontecer nos dados de treino de cada fold!")
except Exception as e:
    print(f"❌ ERRO NO CHECKPOINT 2: {e}")

## PASSO 4: Validação Cruzada Estratificada

### 🔍 DIAGNÓSTICO: Por que Stratified K-Fold?
Com ~38% de sobreviventes, um K-Fold aleatório pode criar folds com 20% ou 55% de positivos. Isso torna a estimativa do AUC instável e pouco confiável.

⚠️ **INSTRUÇÃO:**
1. Instancie um `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`
2. Use `cross_val_score` com `scoring='roc_auc'` para avaliar o `pipeline` em `X_train` / `y_train`
3. Calcule e exiba a média e o desvio padrão dos scores

In [ ]:
# --- SEU CÓDIGO AQUI ---

# 1. Instancie o StratifiedKFold
cv_estratificado = None

# 2. Execute a validação cruzada com cross_val_score
# Dica: cross_val_score(estimator, X, y, cv=..., scoring='roc_auc')
scores_estratificado = None

# 3. Calcule média e desvio padrão
media = None
desvio = None

# --- FIM DO CÓDIGO ---

# --- CHECKPOINT 3 ---
try:
    assert cv_estratificado is not None, "Instancie o StratifiedKFold antes de continuar!"
    assert scores_estratificado is not None, "Execute o cross_val_score antes de continuar!"
    assert media is not None and desvio is not None, "Calcule média e desvio padrão!"
    assert len(scores_estratificado) == 5,         f"Esperado 5 scores (um por fold), encontrado: {len(scores_estratificado)}"
    assert 0.5 < media < 1.0,         f"AUC médio fora do esperado: {media:.4f}. Verifique o pipeline e o scoring."
    assert media > 0.70,         f"AUC médio muito baixo ({media:.4f}). Verifique se o pipeline está correto."
    print("✅ CHECKPOINT 3 PASSOU! Validação cruzada executada com sucesso.")
    print(f"\n📊 Resultados — StratifiedKFold(5):")
    print(f"   AUC por fold:  {scores_estratificado.round(3)}")
    print(f"   AUC médio:     {media:.4f}")
    print(f"   Desvio padrão: {desvio:.4f}")
    print(f"\n💡 Interprete o resultado:")
    print(f"   AUC {media:.2f} significa que o modelo discrimina corretamente")
    print(f"   {media:.0%} dos pares (sobreviveu, não sobreviveu).")
except Exception as e:
    print(f"❌ ERRO NO CHECKPOINT 3: {e}")

## PASSO 5: Comparação — K-Fold Simples vs. Stratified K-Fold

### 🔍 DIAGNÓSTICO: O que queremos ver?
Já calculamos o AUC com `StratifiedKFold`. Agora vamos calcular com `KFold` simples (sem estratificação) e comparar os resultados. A diferença nos mostrará por que a escolha da estratégia de validação importa.

⚠️ **INSTRUÇÃO:**
1. Instancie um `KFold(n_splits=5, shuffle=True, random_state=42)`
2. Execute `cross_val_score` com o **mesmo pipeline** e `scoring='roc_auc'`
3. Compare com os scores do Passo 4

In [ ]:
# --- SEU CÓDIGO AQUI ---

# 1. Instancie o KFold simples (sem estratificação)
cv_simples = None

# 2. Execute cross_val_score com KFold simples
scores_simples = None

# --- FIM DO CÓDIGO ---

# --- CHECKPOINT 4 ---
try:
    assert cv_simples is not None, "Instancie o KFold antes de continuar!"
    assert scores_simples is not None, "Execute o cross_val_score antes de continuar!"
    assert len(scores_simples) == 5,         f"Esperado 5 scores, encontrado: {len(scores_simples)}"

    print("✅ CHECKPOINT 4 PASSOU! Comparação realizada.")
    print()
    print("=" * 56)
    print("    KFold Simples vs. StratifiedKFold — Titanic")
    print("=" * 56)
    print(f"  KFold simples   — AUC: {scores_simples.mean():.4f} ± {scores_simples.std():.4f}")
    print(f"  StratifiedKFold — AUC: {scores_estratificado.mean():.4f} ± {scores_estratificado.std():.4f}")
    diff_desvio = scores_simples.std() - scores_estratificado.std()
    print(f"\n  Diferença no desvio padrão: {diff_desvio:+.4f}")
    print("=" * 56)
    print()
    if scores_simples.std() > scores_estratificado.std():
        print("📌 O KFold simples gerou MAIOR variância entre os folds.")
        print("   Isso confirma: Stratified K-Fold é mais estável em datasets")
        print("   desbalanceados — a estimativa do AUC é mais confiável.")
    else:
        print("📌 Neste dataset, a diferença foi pequena.")
        print("   Em datasets mais desbalanceados, o impacto é muito maior!")
except Exception as e:
    print(f"❌ ERRO NO CHECKPOINT 4: {e}")

### 🤔 Reflexão Final

Responda mentalmente (ou no chat durante a aula):

1. **Por que usamos `stratify=y` no `train_test_split`?**
   > Porque preserva a proporção das classes em treino e teste, evitando que o conjunto de teste seja não-representativo.

2. **Por que o Pipeline previne Data Leakage Tipo 03?**
   > Porque o `fit()` de cada transformador ocorre apenas nos dados de treino de cada fold. O `X_test` nunca influencia os parâmetros aprendidos.

3. **Quando você usaria `GroupKFold` em vez de `StratifiedKFold`?**
   > Quando os dados têm entidades repetidas — como múltiplas internações do mesmo paciente — que não podem aparecer em treino e teste simultaneamente.
